In [2]:
import pandas as pd
import numpy as np
import json

In [2]:
leaderboard = pd.read_parquet(r"E:\projects\HlTraderAction\output\hyperliquidLeaderboard.parquet")
leaderboard.info()

<class 'pandas.DataFrame'>
RangeIndex: 40391 entries, 0 to 40390
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ethAddress    40391 non-null  str    
 1   accountValue  40391 non-null  float64
 2   prize         40391 non-null  int64  
 3   displayName   1397 non-null   str    
 4   day_pnl       40391 non-null  float64
 5   day_roi       40391 non-null  float64
 6   day_vlm       40391 non-null  float64
 7   week_pnl      40391 non-null  float64
 8   week_roi      40391 non-null  float64
 9   week_vlm      40391 non-null  float64
 10  month_pnl     40391 non-null  float64
 11  month_roi     40391 non-null  float64
 12  month_vlm     40391 non-null  float64
 13  allTime_pnl   40391 non-null  float64
 14  allTime_roi   40391 non-null  float64
 15  allTime_vlm   40391 non-null  float64
dtypes: float64(13), int64(1), str(2)
memory usage: 6.6 MB


In [3]:
df = leaderboard.copy()

avP70, avP95 = df["accountValue"].quantile([0.70, 0.95]).values
df["scaleTier"] = np.select(
    [df["accountValue"] >= avP95, df["accountValue"] >= avP70],
    ["whale", "midTier"],
    default="longTail",
)

df["isRecentlyActive"] = df["month_vlm"] > 0
df["isHistoricallyActive"] = df["allTime_vlm"] > 0

conditions = [
    df["isRecentlyActive"] & df["scaleTier"].isin(["whale", "midTier"]),
    df["isRecentlyActive"] & (df["scaleTier"] == "longTail"),
    (~df["isRecentlyActive"]) & df["scaleTier"].isin(["whale", "midTier"]),
    (~df["isRecentlyActive"]) & (df["scaleTier"] == "longTail") & df["isHistoricallyActive"],
    (~df["isRecentlyActive"]) & (df["scaleTier"] == "longTail") & (~df["isHistoricallyActive"]),
]
choices = [
    "activeCore",       
    "activeLongTail",   
    "silentHolder",      
    "churnedTrader",    
    "dormantDepositor",  
]
df["statusTier"] = np.select(conditions, choices, default="edgeCase")

print(df["statusTier"].value_counts())
print(df.groupby("statusTier")[["accountValue", "allTime_vlm", "month_vlm", "allTime_roi"]].median())

statusTier
churnedTrader     18843
activeLongTail     9430
activeCore         6656
silentHolder       5462
Name: count, dtype: int64
                 accountValue   allTime_vlm    month_vlm  allTime_roi
statusTier                                                           
activeCore      241840.643513  1.678801e+07  1161434.995     0.257652
activeLongTail    3355.439251  2.544659e+07   635705.660    -0.385298
churnedTrader        0.568490  2.651868e+07        0.000    -0.477762
silentHolder    260474.794036  2.305766e+05        0.000     0.296517


In [4]:
minVlmForRoiSample = 1_000_000

meaningfulPool = df[df["allTime_vlm"] >= minVlmForRoiSample]

extremeRoi = pd.concat([
    meaningfulPool.nsmallest(15, "allTime_roi"),
    meaningfulPool.nlargest(15, "allTime_roi"),
])

In [5]:
def secondarySample(pool, size, sortCol, ascending=None):
    if len(pool) <= size:
        return pool
    pool = pool.sort_values(sortCol, ascending=True)
    half = size // 2
    return pd.concat([pool.head(half), pool.tail(size - half)]).drop_duplicates()

samples = []

pool = df[df["statusTier"] == "activeCore"]
samples.append(secondarySample(pool, 100, "allTime_roi"))

pool = df[df["statusTier"] == "activeLongTail"]
samples.append(secondarySample(pool, 80, "allTime_vlm"))

pool = df[df["statusTier"] == "churnedTrader"]
vlmP90 = pool["allTime_vlm"].quantile(0.90)
bigChurned = pool[pool["allTime_vlm"] >= vlmP90]        
regularChurned = pool[pool["allTime_vlm"] < vlmP90]      
samples.append(secondarySample(bigChurned, 40, "allTime_roi"))
samples.append(secondarySample(regularChurned, 60, "allTime_roi"))

pool = df[df["statusTier"] == "silentHolder"]
samples.append(secondarySample(pool, 40, "accountValue"))

finalSample = pd.concat(samples).drop_duplicates(subset="ethAddress")
print(f"Final sample amount: {len(finalSample)}")
print(finalSample["statusTier"].value_counts())

finalSample[["ethAddress", "statusTier", "accountValue", "allTime_vlm", "allTime_roi", "allTime_pnl"]].to_parquet(
    "../output/addresses_sample.parquet", index=False
)
addressList = finalSample["ethAddress"].tolist()
with open("../output/addresses_sample.json", "w") as f:
    json.dump(addressList, f, indent=2)

Final sample amount: 320
statusTier
activeCore        100
churnedTrader     100
activeLongTail     80
silentHolder       40
Name: count, dtype: int64


In [3]:
sample = pd.read_parquet(r"E:\projects\HlTraderAction\output\addresses_sample.parquet")

In [4]:
sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ethAddress    320 non-null    str    
 1   statusTier    320 non-null    str    
 2   accountValue  320 non-null    float64
 3   allTime_vlm   320 non-null    float64
 4   allTime_roi   320 non-null    float64
 5   allTime_pnl   320 non-null    float64
dtypes: float64(4), str(2)
memory usage: 32.1 KB
